
# ENEC A-001 Agent Skills
## Progressive Disclosure with Markdown Skill Files

This notebook demonstrates a cleaner enterprise pattern:

```text
User Question
     ↓
Read Skills Catalog
     ↓
Select One Skill
     ↓
Load That Skill's Markdown
     ↓
Expose Only Allowed Tools
     ↓
Execute
```

The detailed skill instructions are stored outside the notebook.

### Recommended Databricks structure

```text
/Workspace/Nuclear_Enterprise_360/
│
├── A001 Documents/
│   └── 9 PDF files
│
├── Agent_Skills/
│   ├── skills_catalog.md
│   ├── asset_summary.md
│   ├── approved_procedure.md
│   └── reliability_review.md
│
└── Notebooks/
    └── A001_Progressive_Disclosure_Agent
```

### Why separate Markdown files?

- Easier to maintain
- Easier to govern
- Smaller prompts
- Skills can be updated independently
- The agent loads only what it needs



# 1. Databricks Paths

After uploading the Markdown files into `Agent_Skills`, use these paths.


In [ ]:

SKILLS_FOLDER = "/Workspace/Nuclear_Enterprise_360/Agent_Skills"

SKILLS_CATALOG = f"{SKILLS_FOLDER}/skills_catalog.md"

DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/A001 Documents"

ASSET_TABLE = "workspace.nuclear_enterprise_360.asset_360"

print("Skills folder :", SKILLS_FOLDER)
print("Catalog       :", SKILLS_CATALOG)
print("Documents     :", DOCUMENT_FOLDER)
print("SQL table     :", ASSET_TABLE)



# 2. Read Markdown Skill Files

Databricks Workspace files can be read from `/Workspace/...`.

We use a very small helper.


In [ ]:

from pathlib import Path

def read_skill_file(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Skill file not found: {path}"
        )

    return path.read_text(encoding="utf-8")



# 3. Level 1 — Read Only the Skills Catalog

This is the first level of progressive disclosure.

The full instructions for every skill are **not** loaded yet.


In [ ]:

catalog_text = read_skill_file(SKILLS_CATALOG)

print(catalog_text)



At this point the agent only knows:

- which skills exist,
- what each skill is for.

It does not yet know the detailed workflow.



# 4. Skill File Mapping

Map the selected skill to its detailed Markdown file.


In [ ]:

SKILL_FILES = {
    "asset_summary":
        f"{SKILLS_FOLDER}/asset_summary.md",

    "approved_procedure":
        f"{SKILLS_FOLDER}/approved_procedure.md",

    "reliability_review":
        f"{SKILLS_FOLDER}/reliability_review.md"
}



# 5. Level 2 — Load One Skill Only

The detailed instructions are loaded only after skill selection.


In [ ]:

def load_skill(skill_name):

    if skill_name not in SKILL_FILES:
        raise ValueError(
            f"Unknown skill: {skill_name}"
        )

    path = SKILL_FILES[skill_name]

    return read_skill_file(path)


In [ ]:

selected_skill = "approved_procedure"

skill_instructions = load_skill(selected_skill)

print(skill_instructions)



Notice what happened:

```text
Before selection:
3 short skill descriptions

After selection:
Only approved_procedure.md is loaded
```

The SQL skill and full reliability-review instructions remain undisclosed.



# 6. Simple Skill Router

For teaching, start with a deterministic router.

Later you can replace it with an LLM planner.


In [ ]:

def select_skill(question):

    q = question.lower()

    review_terms = [
        "investigate",
        "reliability review",
        "escalate",
        "assessment",
        "recommend",
        "evidence support",
        "should"
    ]

    if any(term in q for term in review_terms):
        return "reliability_review"

    procedure_terms = [
        "procedure",
        "approved",
        "guidance",
        "version",
        "document"
    ]

    if any(term in q for term in procedure_terms):
        return "approved_procedure"

    return "asset_summary"


In [ ]:

questions = [
    "What is the health score of A-001?",
    "Which inspection procedure is currently approved?",
    "Does the evidence support escalation for reliability review?"
]

for question in questions:
    print(question)
    print("Selected skill:", select_skill(question))
    print()



# 7. SQL Tool

This is a controlled, read-only tool.

The LLM does not receive unrestricted SQL access.


In [ ]:

def sql_tool(asset_id="A-001"):

    safe_asset_id = asset_id.replace("'", "")

    query = f'''
    SELECT
        asset_id,
        asset_name,
        criticality,
        health_score,
        risk_level,
        open_work_orders,
        high_priority_open_work,
        follow_up_findings
    FROM {ASSET_TABLE}
    WHERE asset_id = '{safe_asset_id}'
    LIMIT 1
    '''

    rows = spark.sql(query).collect()

    if not rows:
        return {
            "query": query,
            "result": None
        }

    return {
        "query": query,
        "result": rows[0].asDict()
    }



# 8. Plug In Your Existing RAG Tool

This notebook assumes you already built these functions in the earlier A-001 notebook:

```python
governed_document_tool(question, top_k=6)
call_llm(prompt)
```

For a workshop, you can either:

1. copy those functions into this notebook, or
2. keep them in a reusable Python file and import them.

The second option is cleaner for production.



# 9. Execute Skills

The skill file becomes part of the prompt **only for the selected capability**.


In [ ]:

def execute_asset_summary(question, asset_id="A-001"):

    skill_text = load_skill("asset_summary")

    evidence = sql_tool(asset_id)

    prompt = f'''
SKILL INSTRUCTIONS

{skill_text}

USER QUESTION

{question}

SQL EVIDENCE

{evidence["result"]}

Follow the skill instructions exactly.
'''

    return call_llm(prompt)


In [ ]:

def execute_approved_procedure(question):

    skill_text = load_skill(
        "approved_procedure"
    )

    evidence = governed_document_tool(
        question,
        top_k=6
    )

    context = ""

    for item in evidence:

        context += f'''
Source: {item["source"]}
Page: {item["page"]}
Authority: {item["authority"]}

{item["text"]}

'''

    prompt = f'''
SKILL INSTRUCTIONS

{skill_text}

USER QUESTION

{question}

RETRIEVED DOCUMENT EVIDENCE

{context}

Follow the skill instructions exactly.
'''

    return call_llm(prompt)


In [ ]:

def execute_reliability_review(
    question,
    asset_id="A-001"
):

    skill_text = load_skill(
        "reliability_review"
    )

    sql_evidence = sql_tool(asset_id)

    rag_evidence = governed_document_tool(
        question,
        top_k=6
    )

    document_context = ""

    for item in rag_evidence:

        document_context += f'''
Source: {item["source"]}
Page: {item["page"]}
Authority: {item["authority"]}

{item["text"]}

'''

    prompt = f'''
SKILL INSTRUCTIONS

{skill_text}

USER QUESTION

{question}

STRUCTURED SQL EVIDENCE

{sql_evidence["result"]}

DOCUMENT EVIDENCE

{document_context}

Follow the skill instructions exactly.
'''

    return call_llm(prompt)



# 10. One Skill Executor

Only the selected skill is loaded.


In [ ]:

def execute_skill(
    skill_name,
    question,
    asset_id="A-001"
):

    print("Selected skill:", skill_name)
    print(
        "Loaded file:",
        SKILL_FILES[skill_name]
    )
    print()

    if skill_name == "asset_summary":

        return execute_asset_summary(
            question,
            asset_id
        )

    if skill_name == "approved_procedure":

        return execute_approved_procedure(
            question
        )

    if skill_name == "reliability_review":

        return execute_reliability_review(
            question,
            asset_id
        )

    raise ValueError(
        f"Unsupported skill: {skill_name}"
    )



# 11. Final Progressive Disclosure Agent


In [ ]:

def progressive_agent(
    question,
    asset_id="A-001"
):

    # LEVEL 1
    # Catalog is available for discovery.

    selected_skill = select_skill(
        question
    )

    # LEVEL 2
    # Only the selected Markdown file
    # is loaded inside execute_skill.

    answer = execute_skill(
        selected_skill,
        question,
        asset_id
    )

    return {
        "selected_skill":
            selected_skill,

        "answer":
            answer
    }



# 12. Workshop Demo


In [ ]:

result = progressive_agent(
    '''
    Investigate A-001.

    Does the available evidence
    support escalation for
    qualified reliability review?
    '''
)

print(result["answer"])



# 13. What Participants Should Learn

### Tool
A callable function.

Example:

```text
sql_tool
```

### Skill
Instructions for completing a business capability.

Example:

```text
reliability_review.md
```

### Progressive Disclosure
Load only the detailed instructions required for the current task.

### Agent
Selects a capability, loads its instructions, uses approved tools, and produces an evidence-backed result.

---

## Final Mental Model

```text
TOOLS
= what the agent CAN CALL

SKILLS
= how the agent SHOULD PERFORM A CAPABILITY

PROGRESSIVE DISCLOSURE
= reveal only what the current task needs

GOVERNANCE
= what the agent is ALLOWED TO DO
```
